In [ ]:
from dask.distributed import Client, LocalCluster
import dask.array as da
import numpy as np
from sklearn.linear_model import SGDClassifier
import time

cluster = LocalCluster(n_workers=3, threads_per_worker=1, memory_limit="2GB", dashboard_address=":8787")
client = Client(cluster)
print("Dashboard:", client.dashboard_link)

n_samples = 5_000_000
n_features = 50
chunk_size = 100_000

X = da.random.random((n_samples, n_features), chunks=(chunk_size, n_features))
y = da.random.randint(0, 2, size=(n_samples,), chunks=(chunk_size,))

model = SGDClassifier(loss="log_loss", max_iter=5)

start = time.perf_counter()
for i in range(0, n_samples, chunk_size):
    X_chunk = X[i:i+chunk_size].compute()
    y_chunk = y[i:i+chunk_size].compute()
    model.partial_fit(X_chunk, y_chunk, classes=np.array([0, 1]))
elapsed = time.perf_counter() - start

print("Czas trenowania:", round(elapsed, 2), "s")

print("Czas trenowania:", round(elapsed, 2), "s")


C:\Users\pauli\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\distributed\node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 63054 instead
  warnings.warn(


Dashboard: http://127.0.0.1:63054/status
Czas trenowania: 24.6 s
Czas trenowania: 24.6 s


In [9]:
import joblib
from sklearn.metrics import confusion_matrix, classification_report

joblib.dump(model, "model_incremental.pkl")

loaded_model = joblib.load("model_incremental.pkl")

X_test = np.random.rand(100_000, n_features)
y_test = np.random.randint(0, 2, size=100_000)

y_pred = loaded_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:")
print(cm)

print("\nRaport klasyfikacji:")
print(classification_report(y_test, y_pred))


Confusion matrix:
[[46079  3944]
 [46223  3754]]

Raport klasyfikacji:
              precision    recall  f1-score   support

           0       0.50      0.92      0.65     50023
           1       0.49      0.08      0.13     49977

    accuracy                           0.50    100000
   macro avg       0.49      0.50      0.39    100000
weighted avg       0.49      0.50      0.39    100000



In [ ]:
from sklearn.model_selection import GridSearchCV

X_small = X[:200_000].compute()
y_small = y[:200_000].compute()

param_grid = {
    "loss": ["log_loss", "hinge"],
    "alpha": [0.0001, 0.001, 0.01],
    "penalty": ["l2", "l1"]
}

grid = GridSearchCV(SGDClassifier(max_iter=5), param_grid, cv=3, n_jobs=-1)
grid.fit(X_small, y_small)

print("Najlepsze parametry:")
print(grid.best_params_)

print("\nPorównanie z parametrami z zadania 1:")
print("Zadanie 1: loss='log_loss', alpha=0.0001, penalty='l2'")


Najlepsze parametry:
{'alpha': 0.001, 'loss': 'hinge', 'penalty': 'l1'}

Porównanie z parametrami z zadania 1:
Zadanie 1: loss='log_loss', alpha=0.0001, penalty='l2'


C:\Users\pauli\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
